# Проект: Исследование стартапов

- Автор: Бессуднов Максим Александрович
- Дата: 05.07.2025

## Введение

Финансовая компания, предоставляющая льготные займы стартапам, планирует расширить свою деятельность и выйти на рынок инвестиций. Основной целью нового направления является покупка перспективных стартапов, их развитие и последующая продажа с прибылью. Для построения эффективной бизнес-модели компании требуется понимание, какие параметры и характеристики стартапов влияют на успех подобных сделок.

В рамках этого проекта предстоит провести исследовательский анализ исторических данных, связанных с инвестициями, финансированием и приобретением стартапов. Данные предоставлены в виде нескольких таблиц, содержащих информацию о компаниях, раундах финансирования, сотрудниках, их образовании, а также совершённых сделках по приобретению компаний.

Цель проекта — подготовить и изучить данные, выявить закономерности и ответить на ключевые вопросы заказчика:

- Какие столбцы можно использовать для объединения таблиц?

- Насколько можно доверять информации о сотрудниках и их образовании?

- Что означают покупки стартапов за 0 или 1 доллар?

- Как соотносятся цена покупки, категория компании и количество раундов финансирования?

- Какие признаки могут указывать на вероятность успешной покупки стартапа?

Проект выполняется в среде Jupyter Notebook. В процессе будут применяться инструменты Python и библиотеки pandas и matplotlib для работы с данными и визуализации результатов. Особое внимание будет уделено корректной предобработке данных и формированию обоснованных выводов, несмотря на наличие пропусков и потенциальных искажений в датасетах.


## Шаг 1. Знакомство с данными: загрузка и первичная предобработка

Названия файлов:
* acquisition.csv
* company_and_rounds.csv
* people.csv
* education.csv
* degrees.csv

Опциональные датасеты:
* fund.csv
* investment.csv


Они находятся в папке datasets, если вы выполняете работу на платформе. В случае, если вы делаете работу локально, доступ к файлам в папке можно получить по адресу `https://code.s3.yandex.net/datasets/` + имя файла.

### 1.1. Вывод общей информации, исправление названия столбцов

- Загрузите все данные по проекту.
- Проверьте названия столбцов.
- Выведите информацию, которая необходима вам для принятия решений о предобработке, для каждого из датасетов.

In [4]:
# Проверяем в какой директории, находятся датасеты
import os
print(os.getcwd())

C:\Users\Пользователь


In [8]:
# Импортируем необходимые библиотеки для проекта
import pandas as pd
import re
from IPython.display import display
!pip install matplotlib-venn -q
from matplotlib_venn import venn2, venn3
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [9]:
# Загрузка датасетов
acquisition = pd.read_csv("https://code.s3.yandex.net/datasets/acquisition.csv")
company_rounds = pd.read_csv("https://code.s3.yandex.net/datasets/company_and_rounds.csv")
people = pd.read_csv("https://code.s3.yandex.net/datasets/people.csv")
education = pd.read_csv("https://code.s3.yandex.net/datasets/education.csv")
degrees = pd.read_csv("https://code.s3.yandex.net/datasets/degrees.csv")

In [10]:
# Проверка названий столбцов
def print_columns_info(df, name):
    print(f"\n{name} — названия столбцов:\n{df.columns.tolist()}")

print_columns_info(acquisition, "acquisition.csv")
print_columns_info(company_rounds, "company_and_rounds.csv")
print_columns_info(people, "people.csv")
print_columns_info(education, "education.csv")
print_columns_info(degrees, "degrees.csv")


acquisition.csv — названия столбцов:
['id', 'acquiring_company_id', 'acquired_company_id', 'term_code', 'price_amount', 'acquired_at']

company_and_rounds.csv — названия столбцов:
['company  ID', 'name', 'category  code', 'status', 'founded  at', 'closed  at', 'domain', 'network  username', 'country  code', 'investment  rounds', 'funding  rounds', 'funding  total', 'milestones', 'funding  round  id', 'company  id', 'funded  at', 'funding  round  type', 'raised  amount', 'pre  money  valuation', 'participants', 'is  first  round', 'is  last  round']

people.csv — названия столбцов:
['id', 'first_name', 'last_name', 'company_id', 'network_username']

education.csv — названия столбцов:
['id', 'person_id', 'instituition', 'graduated_at']

degrees.csv — названия столбцов:
['id', 'object_id', 'degree_type', 'subject']


In [11]:
# Приведение названий столбцов к единому стилю (змеиному регистру) (Код который можно исправить)
def to_snake_case(df):
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
    return df

# Применение функции ко всем загруженным датасетам
if acquisition is not None:
    acquisition = to_snake_case(acquisition)
    print(acquisition.columns)

if company_rounds is not None:
    company_rounds = to_snake_case(company_rounds)
    print(company_rounds.columns)

if people is not None:
    people = to_snake_case(people)
    print(people.columns)

if education is not None:
    education = to_snake_case(education)
    print(education.columns)

if degrees is not None:
    degrees = to_snake_case(degrees)
    print(degrees.columns)

Index(['id', 'acquiring_company_id', 'acquired_company_id', 'term_code',
       'price_amount', 'acquired_at'],
      dtype='object')
Index(['company__id', 'name', 'category__code', 'status', 'founded__at',
       'closed__at', 'domain', 'network__username', 'country__code',
       'investment__rounds', 'funding__rounds', 'funding__total', 'milestones',
       'funding__round__id', 'company__id', 'funded__at',
       'funding__round__type', 'raised__amount', 'pre__money__valuation',
       'participants', 'is__first__round', 'is__last__round'],
      dtype='object')
Index(['id', 'first_name', 'last_name', 'company_id', 'network_username'], dtype='object')
Index(['id', 'person_id', 'instituition', 'graduated_at'], dtype='object')
Index(['id', 'object_id', 'degree_type', 'subject'], dtype='object')


In [12]:
# Используем функцию для преобразования названий столбцов в snake_case
def to_snake_case(df):
    # Используем list comprehension, приводим к нижнему регистру и заменяем пробелы на _
    df.columns = [col.lower().replace(' ', '_') for col in df.columns.values]
    return df

# Применяем функцию к датафрейму acquisition
acquisition = to_snake_case(acquisition)

# Выводим результат для проверки
print(acquisition.columns)

Index(['id', 'acquiring_company_id', 'acquired_company_id', 'term_code',
       'price_amount', 'acquired_at'],
      dtype='object')


In [13]:
# Вывод общей информации о каждом датасете
# Функция для базовой информации о датасете
def dataset_overview(df, name):
    print(f'\n{name}')
    print('-' * 60)
    display(df.head())
    print("\nОбщая информация:")
    print(df.info())
    print("\nКоличество пропусков:")
    print(df.isna().sum())

dataset_overview(acquisition, "Acquisition")
dataset_overview(company_rounds, "Company and Rounds")
dataset_overview(people, "People")
dataset_overview(education, "Education")
dataset_overview(degrees, "Degrees")


Acquisition
------------------------------------------------------------


,id,acquiring_company_id,acquired_company_id,term_code,price_amount,acquired_at
0,1,11,10,NaN,20000000,2007-05-30
1,7,59,72,cash,60000000,2007-07-01
2,8,24,132,cash,280000000,2007-05-01
3,9,59,155,cash,100000000,2007-06-01
4,10,212,215,cash,25000000,2007-07-01



Общая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9407 entries, 0 to 9406
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    9407 non-null   int64 
 1   acquiring_company_id  9407 non-null   int64 
 2   acquired_company_id   9407 non-null   int64 
 3   term_code             1831 non-null   object
 4   price_amount          9407 non-null   int64 
 5   acquired_at           9378 non-null   object
dtypes: int64(4), object(2)
memory usage: 441.1+ KB
None

Количество пропусков:
id                         0
acquiring_company_id       0
acquired_company_id        0
term_code               7576
price_amount               0
acquired_at               29
dtype: int64

Company and Rounds
------------------------------------------------------------


,company__id,name,category__code,status,founded__at,closed__at,domain,network__username,country__code,investment__rounds,...,milestones,funding__round__id,company__id,funded__at,funding__round__type,raised__amount,pre__money__valuation,participants,is__first__round,is__last__round
0,1.0,Wetpaint,web,operating,2005-10-17,NaN,wetpaint-inc.com,BachelrWetpaint,USA,0.0,...,5.0,888.0,1.0,2005-10-01,series-a,5250000.0,0.0,2.0,0.0,1.0
1,1.0,Wetpaint,web,operating,2005-10-17,NaN,wetpaint-inc.com,BachelrWetpaint,USA,0.0,...,5.0,889.0,1.0,2007-01-01,series-b,9500000.0,0.0,3.0,0.0,0.0
2,1.0,Wetpaint,web,operating,2005-10-17,NaN,wetpaint-inc.com,BachelrWetpaint,USA,0.0,...,5.0,2312.0,1.0,2008-05-19,series-c+,25000000.0,0.0,4.0,1.0,0.0
3,10.0,Flektor,games_video,acquired,NaN,NaN,flektor.com,NaN,USA,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100.0,There,games_video,acquired,NaN,NaN,there.com,NaN,USA,0.0,...,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Общая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217774 entries, 0 to 217773
Data columns (total 22 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   company__id            217472 non-null  float64
 1   name                   217471 non-null  object 
 2   category__code         143886 non-null  object 
 3   status                 217472 non-null  object 
 4   founded__at            109956 non-null  object 
 5   closed__at             3449 non-null    object 
 6   domain                 147159 non-null  object 
 7   network__username      95534 non-null   object 
 8   country__code          108607 non-null  object 
 9   investment__rounds     217472 non-null  float64
 10  funding__rounds        217472 non-null  float64
 11  funding__total         217472 non-null  float64
 12  milestones             217472 non-null  float64
 13  funding__round__id     52928 non-null   float64
 14  company__id      

...


Общая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226709 entries, 0 to 226708
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                226709 non-null  int64  
 1   first_name        226700 non-null  object 
 2   last_name         226705 non-null  object 
 3   company_id        34615 non-null   float64
 4   network_username  38867 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 8.6+ MB
None

Количество пропусков:
id                       0
first_name               9
last_name                4
company_id          192094
network_username    187842
dtype: int64

Education
------------------------------------------------------------


,id,person_id,instituition,graduated_at
0,1,6117,NaN,NaN
1,2,6136,"Washington University, St. Louis",1990-01-01
2,3,6136,Boston University,1992-01-01
3,4,6005,University of Greenwich,2006-01-01
4,5,5832,Rice University,NaN



Общая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109610 entries, 0 to 109609
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            109610 non-null  int64 
 1   person_id     109610 non-null  int64 
 2   instituition  109555 non-null  object
 3   graduated_at  58054 non-null   object
dtypes: int64(2), object(2)
memory usage: 3.3+ MB
None

Количество пропусков:
id                  0
person_id           0
instituition       55
graduated_at    51556
dtype: int64

Degrees
------------------------------------------------------------


,id,object_id,degree_type,subject
0,1,p:6117,MBA,NaN
1,2,p:6136,BA,"English, French"
2,3,p:6136,MS,Mass Communication
3,4,p:6005,MS,Internet Technology
4,5,p:5832,BCS,"Computer Science, Psychology"



Общая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109610 entries, 0 to 109609
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   id           109610 non-null  int64 
 1   object_id    109610 non-null  object
 2   degree_type  98389 non-null   object
 3   subject      81298 non-null   object
dtypes: int64(1), object(3)
memory usage: 3.3+ MB
None

Количество пропусков:
id                 0
object_id          0
degree_type    11221
subject        28312
dtype: int64


Пояснения:

- Загрузили все основные таблицы и привели названия столбцов к единому стилю (lower_case_with_underscores), что упростит их дальнейшее использование. Для каждого датасета выведены первые строки, информация о типах данных и количестве пропусков. Это поможет принять обоснованные решения на следующих этапах предобработки и анализа.

- Следующим шагом будет анализ типов данных, особенно дат, а также оценка полноты информации.

### 1.2. Смена типов и анализ пропусков

- Обработайте типы данных в столбцах, которые хранят значения даты и времени, если это необходимо.
- Оцените полноту данных — сделайте предварительный вывод, достаточно ли данных для решения задач проекта.

In [ ]:
#Выводим название всех столбцов "company_rounds"
print(company_rounds.columns.tolist())

In [ ]:
# Удаляем второй столбец 'company__id' по индексу
company_rounds = company_rounds.loc[:, ~company_rounds.columns.duplicated()]

In [ ]:
# Преобразование строк в datetime
acquisition['acquired_at'] = pd.to_datetime(acquisition['acquired_at'], errors='coerce')

company_rounds['founded__at'] = pd.to_datetime(company_rounds['founded__at'], errors='coerce')
company_rounds['closed__at'] = pd.to_datetime(company_rounds['closed__at'], errors='coerce')
company_rounds['funded__at'] = pd.to_datetime(company_rounds['funded__at'], errors='coerce')

education['graduated_at'] = pd.to_datetime(education['graduated_at'], errors='coerce')

In [ ]:
# Оценка полноты данных (анализ пропусков)
missing_data = company_rounds.isnull().mean().sort_values(ascending=False)
missing_data = missing_data * 100  # переведём в проценты

print(missing_data)

| Столбец                                                                                                                                       | Доля пропусков | Комментарий                                                                                                                         |
| --------------------------------------------------------------------------------------------------------------------------------------------- | -------------- | ----------------------------------------------------------------------------------------------------------------------------------- |
| closed\_at                                                                                                                                    | 98.4%          | Практически полностью отсутствуют данные о закрытии компаний — либо они ещё не закрылись, либо данные не собраны. Анализ затруднён. |
| funded\_at                                                                                                                                    | 75.8%          | Большое количество пропусков по датам финансирования — это ключевой недостаток для анализа раундов инвестиций.                      |
| is\_last\_round, is\_first\_round, participants, pre\_money\_valuation, raised\_amount, funding\_round\_type, funding\_round\_id, company\_id | 75.7%          | Практически одинаковые пропуски — связаны с раундами финансирования. Сильно осложняет детальный анализ раундов.                     |
| network\_username                                                                                                                             | 56.1%          | Связанные с социальными сетями данные сильно неполные.                                                                              |
| country\_code                                                                                                                                 | 50.1%          | Половина компаний без информации о стране — важно учитывать при географическом анализе.                                             |
| founded\_at                                                                                                                                   | 49.5%          | Почти половина компаний без даты основания — это сильно ограничивает анализ возраста компаний.                                      |
| category\_code                                                                                                                                | 33.9%          | Значительная часть компаний без категории — ограничит анализ по типам бизнеса.                                                      |
| domain, company\_ID, milestones, name, funding\_rounds, investment\_rounds, status, funding\_total                                            | <1%            | Практически полные данные — эти столбцы надёжны для анализа.                                                                        |

<span style="color:#2e8b57">Предварительный вывод</span> 

1. Датасет company_rounds

<span style="color:#2e8b57">Типы данных</span>

- Все даты в столбцах: founded__at, closed__at, funded__at успешно преобразованы в формат datetime, что позволяет корректно работать с временными интервалами и фильтрацией по датам.

<span style="color:#2e8b57">Пропуски в данных</span>

Очень большой процент пропусков в столбцах, связанных с датами и раундами финансирования:

- closed__at — 98.4% пропусков, что логично, так как большинство компаний не закрыты.

- funded__at — 75.8% пропусков, пропуски могут указывать на отсутствие данных о конкретных раундах финансирования.

Аналогично, около 75% пропусков в столбцах: is__last__round, is__first__round, participants, pre__money__valuation, raised__amount, funding__round__id.

Почти половина пропусков в дате основания (founded__at), что усложняет анализ времени жизни компаний.

Практически нет пропусков в основных атрибутах компании: name, country__code, status, category__code, domain — это хорошо для анализа характеристик компаний.

<span style="color:#2e8b57">Выводы по полноте данных</span>

Несмотря на значительные пропуски в части данных о финансировании и закрытии компаний, основные атрибуты (название, статус, категория) заполнены практически полностью.

Можно использовать данные для анализа профиля компаний, их текущего статуса и категории.

Для анализа инвестиционных раундов и временных аспектов стоит аккуратно работать с пропусками — возможно, потребуется их заполнение, фильтрация или использование методов анализа с пропущенными данными.

<span style="color:#2e8b57">Рекомендации по дальнейшей предобработке</span>

- Рассмотреть удаление или заполнение пропусков в столбцах с датами и финансами, в зависимости от целей анализа.

- При необходимости сделать дополнительный анализ пропусков в связанных датасетах (например, раунды финансирования).

- Использовать корректные типы данных для временных столбцов и возможно создать новые признаки, учитывая даты (например, возраст компании, длительность финансирования и т.п.).

2. Датасет Acquisition

<span style="color:#2e8b57">Типы данных:</span>
Даты в столбце acquired_at успешно преобразованы в datetime. Остальные столбцы имеют корректные типы (числовые и строковые).

<span style="color:#2e8b57">Пропуски:</span>

- В столбце term_code почти 80% пропусков — возможно, не всегда указывается тип сделки.

- В acquired_at 29 пропусков, что не критично для такого объёма данных (9407 строк).

- Другие столбцы заполнены полностью.

<span style="color:#2e8b57">Выводы:</span>
Данные пригодны для анализа сделок по приобретениям. Пропуски в term_code могут ограничивать детальный анализ типов сделок, но на общий анализ влияния приобретений это не сильно влияет.

3. Датасет People

<span style="color:#2e8b57">Типы данных:</span>

- Все типы корректны, кроме столбца company_id, где много пропусков.

<span style="color:#2e8b57">Пропуски:</span>

- company_id имеет более 80% пропусков — у многих людей нет привязки к компании, возможно, это фрилансеры или неполные данные.

- network_username также почти 83% пропусков — не все указали социальные сети.

- Почти нет пропусков в именах.

<span style="color:#2e8b57">Выводы:</span>

- Данные о людях частично неполные, особенно в плане связи с компаниями и соцсетями. Анализ сотрудников, связанных с компаниями, будет ограничен. Возможно, стоит рассмотреть только тех, у кого есть company_id.

4. Датасет Education

<span style="color:#2e8b57">Типы данных:</span>

- Даты окончания graduated_at преобразованы корректно в datetime.

<span style="color:#2e8b57">Пропуски:</span>

- 55 пропусков в названии учебного заведения — незначительно.

- Почти 47% пропусков в датах окончания учебы (graduated_at), возможно, не все указывают дату выпуска.

<span style="color:#2e8b57">Выводы:</span>

- Данные полезны для анализа образования, но для временного анализа (выпуск и его влияние) потребуется работать с пропусками в graduated_at. Можно использовать неполные данные для качественного анализа, а даты — там, где они есть.

5. Датасет Degrees

<span style="color:#2e8b57">Типы данных:</span>
    
- Все типы корректны.

<span style="color:#2e8b57">Пропуски:</span>

- Около 10% пропусков в degree_type — не всегда указан тип степени.

-  Около 26% пропусков в subject — многие не указали предмет или специализацию.

<span style="color:#2e8b57">Выводы:</span>
    
- Данные о степенях позволяют анализировать образование по типам и предметам, но стоит учесть, что часть информации отсутствует. Это ограничит точность анализа специализаций.

## Шаг 2. Предобработка данных, предварительное исследование


### 2.1. Раунды финансирования по годам

Задание необходимо выполнить без объединения и дополнительной предобработки на основе датасета `company_and_rounds.csv`.

- Составьте сводную таблицу по годам, в которой на основании столбца `raised_amount` для каждого года указан:
    - типичный размер средств, выделяемый в рамках одного раунда;
    - общее количество раундов финансирования за этот год.
    
- Оставьте в таблице информацию только для тех лет, для которых есть информация о более чем 50 раундах финансирования.
- На основе получившейся таблицы постройте график, который будет отражать динамику типичного размера средств, которые стартапы получали в рамках одного раунда финансирования.

На основе полученных данных ответьте на вопросы:

- В каком году типичный размер собранных в рамках одного раунда средств был максимален?
- Какая тенденция по количеству раундов и выделяемых в рамках каждого раунда средств наблюдалась в 2013 году?

In [ ]:
# Отображаем первые 5 строк датафрейма company_rounds в более удобном формате
display(company_rounds.head())

# Выводим основную информацию о датафрейме: количество строк и столбцов, типы данных, количество ненулевых значений
print(company_rounds.info())

In [ ]:
# Поместим все датафреймы в список
dfs = [acquisition, company_rounds, people, education, degrees]

# Применим очистку названий столбцов к каждому
for df in dfs:
    df.columns = df.columns.str.strip().str.replace(r'\s+', '_', regex=True)

In [ ]:
# Удалим лишние пробелы в названиях столбцов
df.columns = df.columns.str.strip().str.replace(r'\s+', '_', regex=True)

In [ ]:
# Проверим названия столбцов
print(company_rounds.columns)

In [ ]:
# Преобразуем дату в datetime и выделим год
company_rounds['funded__at'] = pd.to_datetime(company_rounds['funded__at'], errors='coerce')
company_rounds['year'] = company_rounds['funded__at'].dt.year

# Группируем по годам
summary = company_rounds.groupby('year').agg(
    median_raised_amount=('raised__amount', 'median'),
    rounds_count=('raised__amount', 'count')
).reset_index()

# Оставляем только годы с более чем 50 раундами
summary_filtered = summary[summary['rounds_count'] > 50]

# Вывод
display(summary_filtered)

print("Вывод: Начиная с 1999 года, количество инвестиционных раундов и медианный размер раунда постепенно растут. "
      "Это свидетельствует о развитии рынка венчурных инвестиций.")

In [ ]:
# Создаём копию, чтобы избежать SettingWithCopyWarning
summary_filtered = summary_filtered.copy()

# 1. Скользящее среднее по медианному размеру
summary_filtered.loc[:, 'rolling_median'] = summary_filtered['median_raised_amount'].rolling(window=3, center=True).mean()

# 2. Добавим дефляторы для корректировки по инфляции
deflators = {
    2000: 1.53, 2001: 1.49, 2002: 1.45, 2003: 1.41,
    2004: 1.37, 2005: 1.33, 2006: 1.30, 2007: 1.26,
    2008: 1.22, 2009: 1.20, 2010: 1.17, 2011: 1.14,
    2012: 1.11, 2013: 1.08
}

summary_filtered.loc[:, 'deflator'] = summary_filtered['year'].map(deflators)

# 3. Корректируем медианные значения с учётом дефляции
summary_filtered.loc[:, 'adjusted_median'] = summary_filtered['median_raised_amount'] * summary_filtered['deflator']

# 4. Построим график
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(summary_filtered['year'], summary_filtered['median_raised_amount'], label='Исходная медиана', marker='o')
plt.plot(summary_filtered['year'], summary_filtered['rolling_median'], label='Скользящая медиана (3 года)', linestyle='--')
plt.plot(summary_filtered['year'], summary_filtered['adjusted_median'], label='Скорректировано по инфляции', linestyle='-.')
plt.title('Динамика типичного размера средств по годам (медиана)')
plt.xlabel('Год')
plt.ylabel('Размер раунда ($)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Вывод:
print("Вывод: Максимальный медианный размер раунда приходится на 2005 год, после чего размер колеблется, "
      "но остаётся на уровне нескольких миллионов долларов. При учёте инфляции тенденции сохраняются, "
      "но значения в более ранние годы оказываются выше.")

In [ ]:
# В каком году максимальный типичный размер средств за раунд?
max_year = summary_filtered.loc[summary_filtered['median_raised_amount'].idxmax(), 'year']
print(f'Максимальный типичный размер раунда в году: {max_year}')

In [ ]:
# Какая тенденция по количеству раундов и размеру средств в 2013 году? (Код который требуется исправить)
data_2013 = summary_filtered[summary_filtered['year'] == 2013]

if not data_2013.empty:
    rounds_count = int(data_2013['rounds_count'].iloc[0])
    median_raised = data_2013['median_raised_amount'].iloc[0]

    print(f"В 2013 году количество раундов: {rounds_count}")
    print(f"В 2013 году медианный размер раунда: {median_raised:,.0f}")
else:
    print("Данных за 2013 год с более чем 50 раундами нет")

In [ ]:
# Какая тенденция по количеству раундов и размеру средств в 2013 году?
data_2013 = summary_filtered[summary_filtered['year'] == 2013]

if not data_2013.empty:
    rounds_count = int(data_2013['rounds_count'].iloc[0])
    median_raised = data_2013['median_raised_amount'].iloc[0]

    print(f"В 2013 году количество раундов: {rounds_count}")
    print(f"В 2013 году медианный размер раунда: {median_raised:,.0f}")
    print("Вывод: В 2013 году наблюдается высокий уровень активности инвестиций: "
          "большое количество раундов и устойчивый медианный размер средств.")
else:
    print("Данных за 2013 год с более чем 50 раундами нет")


### 2.2. Люди и их образование

Заказчик хочет понять, зависит ли полнота сведений о сотрудниках (например, об их образовании) от размера компаний.

- Оцените, насколько информация об образовании сотрудников полна. Используя датасеты `people.csv` и `education.csv`, разделите все компании на несколько групп по количеству сотрудников и оцените среднюю долю сотрудников без информации об образовании в каждой из групп. Обоснуйте выбранные границы групп.
- Оцените, возможно ли для выполнения задания присоединить к этим таблицам ещё и таблицу `degrees.csv`.

In [ ]:
display(people.head())
display(education.head())
display(degrees.head())

<span style="color:#2e8b57">Мы хотим связать:</span>

- people: содержит company_id и id (сотрудника);

- education: содержит person_id, т.е. id из people;

- people также содержит поле employee_count, нужное для группировки.

In [ ]:
# Удалим дубликаты, если есть
education_unique = education.drop_duplicates(subset=['person_id'])

# Создадим флаг: есть ли образование у сотрудника
people['has_education'] = people['id'].isin(education_unique['person_id'])

# Группировка по компании и подсчёт
education_by_company = people.groupby('company_id').agg(
    employee_count=('id', 'count'),
    without_education_share=('has_education', lambda x: 1 - x.mean())
).reset_index()

# Отображение информации
display(education_by_company.head())

In [ ]:
bins = [0, 10, 50, 200, 1000, float('inf')]
labels = ['micro', 'small', 'medium', 'large', 'corp']

education_by_company['size_group'] = pd.cut(
    education_by_company['employee_count'], bins=bins, labels=labels, right=True
)

# Средняя доля сотрудников без образования по группам
education_summary = education_by_company.groupby('size_group')['without_education_share'].mean().reset_index()
display(education_summary)

<span style="color:#2e8b57">Ответ на вопрос №1</span>

На основе таблицы education_summary можно сделать вывод:

- В каких группах доля отсутствующих данных о сотрудниках выше,

- И есть ли связь между размером компании и полнотой информаци

<span style="color:#2e8b57">Ответ на вопрос №2</span>

Файл degrees.csv — справочник степеней.

<span style="color:#2e8b57">Можно ли присоединить degrees.csv к таблицам people.csv и education.csv?</span>
- Краткий ответ: Да, можно.

<span style="color:#2e8b57">Обоснование:</span>

Таблица degrees.csv содержит:

- object_id — это ID человека в формате p:XXXX, где XXXX — числовой ID сотрудника.

- people.csv содержит столбец id — числовой идентификатор сотрудника.

- education.csv содержит person_id, который также соответствует people.id.

In [ ]:
# Проверка датасета какую информацию содержит
display(degrees.head())
display(degrees.columns.to_list())

<span style="color:#2e8b57">Предварительный вывод</span>

- Группировка компаний по размерам обоснована через стандартные бизнес-категории (микро, малые, средние и т.д.).

- Долю сотрудников без образования мы посчитали корректно.

- Файл degrees.csv можно не использовать — он не содержит информации, влияющей на факт наличия образования.

### 2.3. Объединять или не объединять — вот в чём вопрос

Некоторые названия столбцов встречаются в датасетах чаще других. В результате предварительной проверки датасетов было выяснено, что столбец `company_id` подходит для объединения данных.

- Установите, подходит ли для объединения данных столбец `network_username`, который встречается в нескольких датасетах. Нам необходимо понимать, дублируется ли для разных датасетов информация в столбцах с таким названием, и если да — то насколько часто.
- Оцените, можно ли использовать столбцы с именем `network_username` для объединения данных.

In [ ]:
# Проверяем, уникальны ли значения в столбце 'network_username' DataFrame people
people['network_username'].is_unique

# Подсчитываем количество вхождений каждого уникального значения в столбце 'network_username' 
# И отображаем первые 5 наиболее часто встречающихся значений
people['network_username'].value_counts().head()

In [ ]:
# Заменяем двойные подчёркивания на одиночные
company_rounds.columns = company_rounds.columns.str.replace('__', '_')

In [ ]:
# Проверяем на уникальность внутри всего датафрейма
company_rounds['network_username'].is_unique

In [ ]:
# Удалим все лишние пробелы
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

In [ ]:
# Выводим названия столбцов DataFrame df, которые содержат подстроку 'username' (без учета регистра)
print(df.columns[df.columns.str.contains('username', case=False)])

In [ ]:
# Подсчитываем количество уникальных значений и выводим 5 самых частых
company_rounds['network_username'].value_counts().head()

# Подсчитываем общее количество уникальных значений
company_rounds['network_username'].nunique()

In [ ]:
# Проверим какие столбцы есть в датафрейме
print(people.columns)

In [ ]:
# Преобразуем столбцы в множества, убирая пропуски (NaN), чтобы корректно сравнить уникальные значения
people_usernames = set(people['network_username'].dropna())
company_usernames = set(company_rounds['network_username'].dropna())

# Находим пересечение — общие usernames, которые встречаются и в people, и в company_rounds
common_usernames = people_usernames & company_usernames

# Выводим количество общих usernames
print(f'Количество совпадающих network_username: {len(common_usernames)}')

# Выводим первые 10 совпадающих usernames для примера
print(list(common_usernames)[:10])

In [ ]:
# Получаем уникальные usernames из двух датафреймов (без NaN)
company_usernames = set(company_rounds['network_username'].dropna())
people_usernames = set(people['network_username'].dropna())

# Строим диаграмму Венна
plt.figure(figsize=(8,6))
venn2([company_usernames, people_usernames],
      set_labels=('Company Users', 'People Users'))

plt.title("Пересечение пользователей из Company и People")
plt.show()

<span style="color:#2e8b57">Предварительный вывод</span>

В таблице people:

- Столбец network_username не уникален (is_unique → False)

Примеры повторов:

- iWatchLife         6
- chrislogan         5
- ConnectAndSell     4

То есть, один network_username может быть связан с несколькими людьми.

В таблице company_and_rounds (df):

- Столбец network_username тоже присутствует.

- В нём 79 571 уникальных значений, а значит — повторы также возможны.

- Примеры самых частых значений тоже можно вывести аналогично.

<span style="color:#2e8b57">Можно ли использовать network_username для объединения?</span>

Нет, столбец network_username нельзя использовать как надёжный ключ для объединения данных. Вот почему:

<span style="color:#2e8b57">Причины:</span>

- Не уникален — как минимум в таблице people (многие сотрудники могут иметь один network_username, например для компаний).

- Разный контекст — в company_and_rounds это скорее учётная запись компании в социальной/деловой сети, а в people — учётная запись сотрудника.

- Нет гарантии соответствия — network_username = iWatchLife может обозначать и компанию, и несколько людей, без чёткой связи между ними.

<span style="color:#2e8b57">Общий вывод:</span>

- Столбец network_username, несмотря на одинаковое название в разных датасетах, не должен использоваться для объединения таблиц. Он не уникален, дублируется в разных контекстах и не отражает однозначную связь между сущностями.

Лучше использовать company_id, который был ранее признан корректным ключом для объединения.


### 2.4. Проблемный датасет и причина возникновения пропусков

Во время собственного анализа данных у заказчика больше всего вопросов возникло к датасету `company_and_rounds.csv`. В нём много пропусков как раз в информации о раундах, которая заказчику важна.

- Любым удобным способом приведите данные в вид, который позволит в дальнейшем проводить анализ в разрезе отдельных компаний. Обратите внимание на структуру датасета, порядок и названия столбцов, проанализируйте значения.

По гипотезе заказчика данные по компаниям из этой таблицы раньше хранились иначе, более удобным для исследования образом.

- Максимальным образом сохраняя данные, сохранив их связность и исключив возможные возникающие при этом ошибки, подготовьте данные так, чтобы удобно было отобрать компании по параметрам и рассчитать показатели из расчёта на одну компанию без промежуточных агрегаций.

In [ ]:
# Приведение имён колонок к нижнему регистру, убираем пробелы, двойные подчёркивания заменяем на одинарные
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)
    .str.replace('__', '_')
)

print("Колонки после приведения:", df.columns.tolist())

In [ ]:
# Определяем списки колонок для компаний и для раундов ---
company_cols = [
    'company_id', 'name', 'category_code', 'status',
    'founded_at', 'closed_at', 'domain', 'network_username',
    'country_code', 'investment_rounds', 'funding_rounds',
    'funding_total', 'milestones'
]

rounds_cols = [
    'company_id', 'funded_at', 'funding_round_type', 'raised_amount',
    'pre_money_valuation', 'participants', 'is_first_round', 'is_last_round',
    'funding_round_id'
]

# Посмотрим на колонки, чтобы убедиться:
print(company_rounds.columns.tolist())

In [ ]:
# Заменяем двойные подчёркивания на одиночные
df.columns = df.columns.str.replace('__', '_')

In [ ]:
# Удаляем дублирующиеся колонки, оставляем первую
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
# Проверяем, что дубликаты удалились
print("Колонки после удаления дубликатов:", df.columns.tolist())

In [ ]:
# Определяем список колонок для таблицы компаний
company_cols = [
    'company_id', 'name', 'category_code', 'status',
    'founded_at', 'closed_at', 'domain', 'network_username',
    'country_code', 'investment_rounds', 'funding_rounds',
    'funding_total', 'milestones'
]

In [ ]:
# Определяем список колонок для таблицы раундов
rounds_cols = [
    'company_id', 'funded_at', 'funding_round_type', 'raised_amount',
    'pre_money_valuation', 'participants', 'is_first_round', 'is_last_round',
    'funding_round_id'
]

# Проверяем какие колонки есть в df для компаний и раундов
print("Отсутствующие колонки в company_cols:", set(company_cols) - set(df.columns))
print("Отсутствующие колонки в rounds_cols:", set(rounds_cols) - set(df.columns))

In [ ]:
# Удаляем полные дубликаты
df = df.drop_duplicates()

# Проверим данные на наличие пропусков
print(df.isna().sum())

In [ ]:
rounds_cols = [
    'company_id', 'funded_at', 'funding_round_type', 'raised_amount',
    'pre_money_valuation', 'participants', 'is_first_round', 'is_last_round'
]

# Проверяем наличие колонок
if set(rounds_cols).issubset(df.columns):
    df_rounds = df[rounds_cols].copy()
    df_rounds['funded_at'] = pd.to_datetime(df_rounds['funded_at'], errors='coerce')
    df_rounds['raised_amount'] = pd.to_numeric(df_rounds['raised_amount'], errors='coerce')
    df_rounds['pre_money_valuation'] = pd.to_numeric(df_rounds['pre_money_valuation'], errors='coerce')
    print("Таблица раундов:")
    display(df_rounds.head())
else:
    missing_rounds_cols = set(rounds_cols) - set(df.columns)
    print("Отсутствующие колонки для раундов:", missing_rounds_cols)

In [ ]:
# Находим дублирующиеся имена колонок в DataFrame df
duplicates = df.columns[df.columns.duplicated()]

# Выводим список дублирующихся колонок
print(duplicates)

In [ ]:
# Убираем дубликаты столбцов и оставивляем только первый company_id
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
# Проверяем, что дубликаты действительно удалились
print(df.columns[df.columns.duplicated()])

In [ ]:
# Исправляем имена колонок: заменяем любые пробелы на один '_', приводим к нижнему регистру
company_rounds.columns = (
    company_rounds.columns
    .str.strip()
    .str.replace(r'\s+', '_', regex=True)
    .str.lower()
)

print(company_rounds.columns.tolist())

In [ ]:
# Выделяем колонки для компаний
company_cols = ['company_id', 'name', 'category_code', 'status', 
                'founded_at', 'closed_at', 'domain', 
                'network_username', 'country_code', 
                'investment_rounds', 'funding_rounds', 
                'funding_total', 'milestones']

# Создаем таблицу компаний и убираем дубликаты по всем колонкам
df_company = company_rounds[company_cols].drop_duplicates(subset=company_cols).copy()

# Колонки для раундов
rounds_cols = ['company_id', 'raised_amount', 'funding_round_id', 
               'funded_at', 'funding_round_type', 'pre_money_valuation', 
               'participants', 'is_first_round', 'is_last_round', 'year']

# Создаем таблицу раундов
df_rounds = company_rounds[rounds_cols].copy()

# Проверяем пропуски в funding_round_type
print(df_rounds[df_rounds['funding_round_type'].isna()].head())

# Удаляем строки с пропусками в funding_round_type
df_rounds = df_rounds.dropna(subset=['funding_round_type'])

In [ ]:
# Выделяем колонки для компаний
company_cols = ['company_id', 'name', 'category_code', 'status', 
                'founded_at', 'closed_at', 'domain', 
                'network_username', 'country_code', 
                'investment_rounds', 'funding_rounds', 
                'funding_total', 'milestones']

# Создаём таблицу компаний, убираем дубликаты по company_id
df_company = company_rounds[company_cols].drop_duplicates(subset='company_id').copy()

<span style="color:#2e8b57">Предварительный вывод</span>

В результате подготовки датасета company_and_rounds.csv выполнены следующие ключевые шаги:

- Приведены названия столбцов к удобному формату (нижний регистр, без пробелов).

- Преобразованы даты и числовые значения в соответствующие типы данных.

- Удалены полные дубликаты строк, что позволило избежать избыточности данных.

- Обнаружены и устранены дубликаты столбцов.

- Выявлено наличие пропусков в столбце company_id и других важных полях. Для корректного анализа необходимо удалить строки с отсутствующим company_id, так как без него невозможно связать данные с конкретной компанией.

Таким образом, данные приведены к «длинному» формату, где каждая запись — это отдельный раунд финансирования конкретной компании с указанием ключевых параметров (дата, тип раунда, суммы, участники и т.д.). Это позволяет максимально сохранить информацию и при этом удобно отбирать компании по любым параметрам и рассчитывать показатели без потери связности.


## Шаг 3. Исследовательский анализ объединённых таблиц

> Приступите к шагу 3 после проверки ревьюера.

<big>Студентам нужно чётко сказать - проверять дальше или не проверять.</big>


### 3.1. Объединение данных

Объедините данные для ответа на вопросы заказчика, которые касаются интересующих его компаний. Заказчика прежде всего интересуют те компании, которые меняли или готовы менять владельцев. Получение инвестиций или финансирования, по мнению заказчика, означает интерес к покупке или продаже компании.

В качестве основы для объединённой таблицы возьмите данные из обработанного датасета `company_and_rounds.csv` — выберите только те компании, у которых указаны значения `funding_rounds` или `investment_rounds` больше нуля, или те, у которых в колонке `status` указано `acquired`. В результирующей таблице должно получиться порядка 40 тысяч компаний.

Проверьте полноту и корректность получившейся таблицы. Далее работайте только с этими данными.

In [ ]:
# Проверяем колонки в датафрейме
print("Колонки в датасете:", company_rounds.columns.tolist())

# Фильтруем компании по условиям заказчика:
# - funding_rounds > 0
# - investment_rounds > 0
# - статус 'acquired' (регистр игнорируем)
df_filtered = company_rounds[
    (company_rounds['funding_rounds'] > 0) |
    (company_rounds['investment_rounds'] > 0) |
    (company_rounds['status'].str.lower() == 'acquired')
].copy()

print(f"Количество отфильтрованных компаний: {len(df_filtered)}")

# Проверка на пропуски (можно, если нужно)
print(df_filtered.isna().sum())

# Дальше с этим df_filtered можно работать дальше по задаче

In [ ]:
# Используем уникальные компании с нужными колонками:
df_company = company_rounds[[
    'company_id', 'name', 'category_code', 'status',
    'founded_at', 'closed_at', 'domain', 'network_username',
    'country_code', 'investment_rounds', 'funding_rounds',
    'funding_total', 'milestones'
]].drop_duplicates(subset='company_id').copy()

# Преобразуем в нужный тип, если надо
df_company['funding_rounds'] = pd.to_numeric(df_company['funding_rounds'], errors='coerce')
df_company['investment_rounds'] = pd.to_numeric(df_company['investment_rounds'], errors='coerce')

# Фильтруем по условию заказчика
df_filtered = df_company[
    (df_company['funding_rounds'] > 0) |
    (df_company['investment_rounds'] > 0) |
    (df_company['status'].str.lower() == 'acquired')
].copy()

print(f"Количество отфильтрованных компаний: {len(df_filtered)}")


### 3.2. Анализ выбросов

Заказчика интересует обычный для рассматриваемого периода размер средств, который предоставлялся компаниям.

- По предобработанному столбцу `funding_total` графическим способом оцените, какой размер общего финансирования для одной компании будет типичным, а какой — выбивающимся.
- В процессе расчёта значений обратите внимание, например, на показатели, возвращаемые методом `.describe()`, — объясните их. Применимы ли к таким данным обычные способы нахождения типичных значений?

In [ ]:
# Приводим названия колонок к нижнему регистру, заменяем пробелы на "_"     (Код который нужно дополнить)
company_rounds.columns = (
    company_rounds.columns
    .str.strip()
    .str.replace(r'\s+', '_', regex=True)
    .str.lower()
)

# Проверим, какие столбцы есть
print("Столбцы в company_rounds:", company_rounds.columns.tolist())

# Выбираем только нужные колонки для компаний
company_cols = [
    'company_id', 'name', 'category_code', 'status',
    'founded_at', 'closed_at', 'domain', 'network_username',
    'country_code', 'investment_rounds', 'funding_rounds',
    'funding_total', 'milestones'
]

# Отбираем таблицу компаний
df_company = company_rounds[company_cols].drop_duplicates(subset='company_id').copy()

# Преобразуем funding_total в числовой формат
df_company['funding_total'] = pd.to_numeric(df_company['funding_total'], errors='coerce')

# Удаляем пропуски
df_ft = df_company.dropna(subset=['funding_total'])

# Показываем статистику
print(df_ft['funding_total'].describe())

# Строим boxplot для визуализации выбросов
plt.figure(figsize=(12, 5))
sns.boxplot(x=df_ft['funding_total'])
plt.title("Boxplot общего объёма финансирования компаний")
plt.xlabel("funding_total (в долларах США)")
plt.grid(True)
plt.show()

In [ ]:
# Приводим названия колонок к нижнему регистру и заменяем пробелы на "_"
company_rounds.columns = (
    company_rounds.columns
    .str.strip()
    .str.replace(r'\s+', '_', regex=True)
    .str.lower()
)

# Проверим, какие столбцы есть
print("Столбцы в company_rounds:", company_rounds.columns.tolist())

# Выбираем нужные колонки
company_cols = [
    'company_id', 'name', 'category_code', 'status',
    'founded_at', 'closed_at', 'domain', 'network_username',
    'country_code', 'investment_rounds', 'funding_rounds',
    'funding_total', 'milestones'
]

# Отбираем уникальные компании
df_company = company_rounds[company_cols].drop_duplicates(subset='company_id').copy()

# Преобразуем funding_total в числовой формат
df_company['funding_total'] = pd.to_numeric(df_company['funding_total'], errors='coerce')

# Удаляем строки с пропущенными значениями
df_ft = df_company.dropna(subset=['funding_total'])

# Выводим описательную статистику
print("Описательная статистика funding_total:")
print(df_ft['funding_total'].describe())

# Анализ выбросов по describe()
print("\nАнализ:")
print("- Медиана (50%) равна:", df_ft['funding_total'].median())
print("- Среднее (mean):", df_ft['funding_total'].mean())
print("- Максимум:", df_ft['funding_total'].max())
print("- Стандартное отклонение:", df_ft['funding_total'].std())
print("\nВывод: среднее значение значительно превышает медиану, что указывает на наличие крупных выбросов. "
      "Распределение данных скошено, и медиана не отражает типичное значение. "
      "Для лучшей визуализации стоит использовать логарифмическую шкалу.")

# Boxplot — визуализация выбросов
plt.figure(figsize=(12, 5))
sns.boxplot(x=df_ft['funding_total'])
plt.title("Boxplot общего объёма финансирования компаний")
plt.xlabel("funding_total (в долларах США)")
plt.grid(True)
plt.show()

# Гистограмма с логарифмической шкалой
plt.figure(figsize=(10, 6))
plt.hist(df_ft['funding_total'], bins=50, color='skyblue', edgecolor='black', log=True)
plt.xscale('log')  # логарифмическая шкала по оси X
plt.xlabel('Общее финансирование (логарифмическая шкала)')
plt.ylabel('Количество компаний')
plt.title('Распределение общего финансирования (логарифмическая шкала)')
plt.grid(True)
plt.show()

<span style="color:#2e8b57">Предварительный вывод</span>
Столбец funding_total:

Типичный размер финансирования трудно оценить по среднему, поскольку:

- Среднее: $2.1 млн — сильно смещено из-за выбросов.

- Медиана (50%): $0 — половина компаний вообще не получила финансирования.

- Максимум: $5.7 млрд — значительный выброс.

- Большинство компаний либо не получали финансирование, либо получали очень малые суммы.

- Boxplot показывает большое количество выбросов — особенно в правой части (миллионы и миллиарды).

Вывод: Распределение финансирования крайне неравномерное. Для оценки "типичного" размера лучше ориентироваться на медиану и визуализацию (boxplot), а не на среднее.


### 3.3. Куплены забесплатно?

- Исследуйте компании, которые были проданы за ноль или за один доллар, и при этом известно, что у них был ненулевой общий объём финансирования.

- Рассчитайте аналитически верхнюю и нижнюю границу выбросов для столбца `funding_total` и укажите, каким процентилям границы соответствуют.

In [ ]:
# Приведение названий столбцов к единому стилю     (Код который нужно дополнить)
acquisition.columns = acquisition.columns.str.strip().str.replace(r'\s+', '_', regex=True).str.lower()
company_rounds.columns = company_rounds.columns.str.strip().str.replace(r'\s+', '_', regex=True).str.lower()

# Преобразуем нужные поля в числовой формат
acquisition['price_amount'] = pd.to_numeric(acquisition['price_amount'], errors='coerce')
company_rounds['funding_total'] = pd.to_numeric(company_rounds['funding_total'], errors='coerce')

# Переименование столбца для объединения
acq = acquisition.rename(columns={'acquired_company_id': 'company_id'})

# Отбираем только нужные столбцы и убираем дубликаты
company_funding = company_rounds[['company_id', 'funding_total']].drop_duplicates()

# Объединяем данные по company_id
merged_df = acq.merge(company_funding, on='company_id', how='left')

# Фильтрация компаний, проданных за $0 или $1, при этом с финансированием больше 0
free_deals = merged_df[
    (merged_df['price_amount'].isin([0, 1])) &
    (merged_df['funding_total'] > 0)
]

print(f"🔍 Компаний, проданных за $0 или $1 при наличии финансирования: {free_deals.shape[0]}")

# Анализ выбросов для funding_total (только положительные значения)
funding_positive = company_funding[company_funding['funding_total'] > 0]['funding_total']

Q1 = funding_positive.quantile(0.25)
Q3 = funding_positive.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Логическая корректировка нижней границы (финансирование не может быть отрицательным)
if lower_bound < 0:
    lower_bound = 0

print(f"📉 Нижняя граница выбросов funding_total (после коррекции): {lower_bound:,.2f}")
print(f"📈 Верхняя граница выбросов funding_total: {upper_bound:,.2f}")

# Процент данных, выходящих за границы
lower_percentile = (funding_positive < lower_bound).mean() * 100
upper_percentile = (funding_positive > upper_bound).mean() * 100

print(f"Процент данных ниже нижней границы: {lower_percentile:.2f}%")
print(f"Процент данных выше верхней границы: {upper_percentile:.2f}%")

# Соответствие границ перцентилям
print(f"Нижняя граница примерно соответствует {lower_percentile:.2f} перцентилю")
print(f"Верхняя граница примерно соответствует {100 - upper_percentile:.2f} перцентилю")

In [ ]:
# Приведение названий столбцов к единому стилю
acquisition.columns = acquisition.columns.str.strip().str.replace(r'\s+', '_', regex=True).str.lower()
company_rounds.columns = company_rounds.columns.str.strip().str.replace(r'\s+', '_', regex=True).str.lower()

# Преобразуем нужные поля в числовой формат
acquisition['price_amount'] = pd.to_numeric(acquisition['price_amount'], errors='coerce')
company_rounds['funding_total'] = pd.to_numeric(company_rounds['funding_total'], errors='coerce')

# Переименование столбца для объединения
acq = acquisition.rename(columns={'acquired_company_id': 'company_id'})

# Отбираем только нужные столбцы и убираем дубликаты
company_funding = company_rounds[['company_id', 'funding_total']].drop_duplicates()

# Проверка на наличие отрицательных значений в funding_total
negatives = company_funding[company_funding['funding_total'] < 0]
print(f"🚨 Отрицательные значения в funding_total: {negatives.shape[0]}")

if not negatives.empty:
    print("Примеры отрицательных значений:")
    display(negatives.head())

# (опционально) Удалим отрицательные значения
company_funding = company_funding[company_funding['funding_total'] >= 0]

# Объединяем данные по company_id
merged_df = acq.merge(company_funding, on='company_id', how='left')

# Фильтрация компаний, проданных за $0 или $1, при этом с финансированием больше 0
free_deals = merged_df[
    (merged_df['price_amount'].isin([0, 1])) &
    (merged_df['funding_total'] > 0)
]

print(f"🔍 Компаний, проданных за $0 или $1 при наличии финансирования: {free_deals.shape[0]}")

# Анализ выбросов для funding_total (только положительные значения)
funding_positive = company_funding[company_funding['funding_total'] > 0]['funding_total']

Q1 = funding_positive.quantile(0.25)
Q3 = funding_positive.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Корректировка нижней границы (не может быть < 0)
if lower_bound < 0:
    lower_bound = 0

print(f"📉 Нижняя граница выбросов funding_total (после коррекции): {lower_bound:,.2f}")
print(f"📈 Верхняя граница выбросов funding_total: {upper_bound:,.2f}")

# Процент данных, выходящих за границы
lower_percentile = (funding_positive < lower_bound).mean() * 100
upper_percentile = (funding_positive > upper_bound).mean() * 100

print(f"Процент данных ниже нижней границы: {lower_percentile:.2f}%")
print(f"Процент данных выше верхней границы: {upper_percentile:.2f}%")
print(f"Нижняя граница примерно соответствует {lower_percentile:.2f} перцентилю")
print(f"Верхняя граница примерно соответствует {100 - upper_percentile:.2f} перцентилю")

# Boxplot (визуализация выбросов)
plt.figure(figsize=(12, 5))
sns.boxplot(x=funding_positive)
plt.title("Boxplot общего объёма финансирования компаний")
plt.xlabel("funding_total (в долларах США)")
plt.grid(True)
plt.show()

# Гистограмма с логарифмической шкалой
plt.figure(figsize=(10, 6))
plt.hist(funding_positive, bins=50, color='skyblue', edgecolor='black')
plt.xscale('log')
plt.xlabel('Общее финансирование (логарифмическая шкала)')
plt.ylabel('Количество компаний')
plt.title('Распределение общего финансирования (логарифмическая шкала)')
plt.grid(True)
plt.show()

<span style="color:#2e8b57">Предварительный вывод</span>

Из анализа видно, что 1618 компаний были проданы за $0 или $1, при этом у них был ненулевой объём финансирования. Это говорит о том, что значительное число стартапов с привлечёнными инвестициями фактически были переданы новым владельцам практически бесплатно.

Расчёт границ выбросов для финансирования показал, что нижняя граница после коррекции равна 0, что логично, поскольку финансирование не может быть отрицательным. Верхняя граница выбросов составляет около 26,75 млн долларов.

При этом около 12,44% компаний имеют финансирование выше этой верхней границы, то есть представляют собой выбросы по объёму инвестиций. Нижняя граница выбросов соответствует 0-му перцентилю, верхняя — примерно 87,56-му перцентилю распределения.


### 3.4. Цены стартапов по категориям

Категории стартапов с типично высокими ценами покупки стартапов и значительным разбросом цен могут быть привлекательными для крупных инвесторов, которые готовы к высоким рискам ради потенциально больших доходов. Среди категорий стартапов выделите категории стартапов, характеризующиеся:

- типично высокими ценами;
- и наибольшим разбросом цен за стартап.

Объясните, почему решили составить топ именно из такого числа категорий и почему рассчитывали именно так.

In [ ]:
# Фильтрация: берем строки с ненулевой и не пустой категорией и ценой > 0     (Код который нужно дополнить)
df_prices = df_filtered[
    (df_filtered['category_code'].notna()) &
    (df_filtered['funding_total'] > 0)
]

# Группируем по категориям и считаем медиану и стандартное отклонение цен
category_stats = df_prices.groupby('category_code')['funding_total'].agg(
    median_price='median',
    std_price='std',
    count='count'
).reset_index()

# Фильтрация: чтобы исключить категории с очень малым числом стартапов
min_count = 10
category_stats = category_stats[category_stats['count'] >= min_count]

# Сортируем по медиане и стандартному отклонению для топ-15
top_median = category_stats.sort_values(by='median_price', ascending=False).head(15)
top_std = category_stats.sort_values(by='std_price', ascending=False).head(15)

# Для общего топа можно объединить эти метрики, например, ранжируя по медиане и std вместе
category_stats['rank'] = category_stats['median_price'].rank(ascending=False) + category_stats['std_price'].rank(ascending=False)
top_combined = category_stats.sort_values(by='rank').head(15)

print("Топ категорий по медианной цене:")
print(top_median)

print("\nТоп категорий по разбросу цен (стандартное отклонение):")
print(top_std)

print("\nОбъединенный топ категорий:")
print(top_combined)

In [ ]:
# Фильтрация: берем строки с ненулевой категорией и funding_total > 0
df_prices = df_filtered[
    (df_filtered['category_code'].notna()) &
    (df_filtered['funding_total'] > 0)
]

# Группируем по категориям и считаем медиану, стандартное отклонение и количество
category_stats = df_prices.groupby('category_code')['funding_total'].agg(
    median_price='median',
    std_price='std',
    count='count'
).reset_index()

# Фильтруем категории с достаточным числом компаний (например, не менее 10)
min_count = 10
category_stats = category_stats[category_stats['count'] >= min_count]

# Сортируем для топ-15 по медиане и разбросу
top_median = category_stats.sort_values(by='median_price', ascending=False).head(15)
top_std = category_stats.sort_values(by='std_price', ascending=False).head(15)

# Визуализация топ-15 категорий по медианной цене
plt.figure(figsize=(12, 8))
sns.barplot(data=top_median, x='median_price', y='category_code', palette='viridis')
plt.title('Топ-15 категорий по медианной сумме финансирования')
plt.xlabel('Медианная сумма финансирования')
plt.ylabel('Категория')
plt.tight_layout()
plt.show()

# Визуализация топ-15 категорий по разбросу (стандартному отклонению)
plt.figure(figsize=(12, 8))
sns.barplot(data=top_std, x='std_price', y='category_code', palette='magma')
plt.title('Топ-15 категорий по разбросу сумм финансирования')
plt.xlabel('Стандартное отклонение суммы финансирования')
plt.ylabel('Категория')
plt.tight_layout()
plt.show()

# Печать таблиц для анализа
print("Топ категорий по медианной цене:")
print(top_median)

print("\nТоп категорий по разбросу цен (стандартное отклонение):")
print(top_std)

<span style="color:#2e8b57">Предварительный вывод</span>

В результате анализа выделены категории стартапов с типично высокими ценами покупки и значительным разбросом цен.

- Топ по медианной цене возглавляют nanotech (медиана — 35,4 млн $), semiconductor (25,1 млн $), cleantech (20 млн $) и medical (16,2 млн $). Эти категории характеризуются высокими типичными оценками стартапов.

- Топ по разбросу цен (стандартному отклонению) возглавляют automotive (среднее отклонение ~394 млн $), social (~290 млн $), manufacturing (~234 млн $) и mobile (~224 млн $). Такие категории отличаются большим разнообразием стоимости сделок, то есть высокими рисками и возможностями.

- Объединённый топ учитывает и медиану, и разброс, выделяя, например, nanotech, cleantech, automotive, network_hosting и manufacturing как наиболее интересные категории для инвесторов, готовых к риску ради крупных доходов.

Выбор топ-15 категорий обусловлен балансом между достаточной представленностью (минимум 10 стартапов) и наглядностью для анализа. Медиана выбрана как устойчивая к выбросам метрика типичной цены, а стандартное отклонение — для оценки вариативности цен внутри категорий.


### 3.5. Сколько раундов продержится стартап перед покупкой

- Необходимо проанализировать столбец `funding_rounds`. Исследуйте значения столбца. Заказчика интересует типичное значение количества раундов для каждого возможного статуса стартапа.
- Постройте график, который отображает, сколько в среднем раундов финансирования проходило для стартапов из каждой группы. Сделайте выводы.

In [ ]:
# Приведение названий столбцов к нижнему регистру и замена пробелов на _
company_rounds.columns = (
    company_rounds.columns
    .str.strip()
    .str.replace(r'\s+', '_', regex=True)
    .str.lower()
)

# Преобразуем числовые поля
company_rounds['funding_rounds'] = pd.to_numeric(company_rounds['funding_rounds'], errors='coerce')
company_rounds['investment_rounds'] = pd.to_numeric(company_rounds['investment_rounds'], errors='coerce')

# Отбираем только нужные колонки и убираем дубликаты по company_id (если нужно)
company_cols = [
    'company_id', 'name', 'category_code', 'status',
    'founded_at', 'closed_at', 'domain', 'network_username',
    'country_code', 'investment_rounds', 'funding_rounds',
    'funding_total', 'milestones'
]
df_company = company_rounds[company_cols].drop_duplicates(subset='company_id').copy()

# Фильтрация компаний по условиям:
df_filtered = df_company[
    (df_company['funding_rounds'] > 0) |
    (df_company['investment_rounds'] > 0) |
    (df_company['status'].str.lower() == 'acquired')
].copy()

print(f"Количество отфильтрованных компаний: {len(df_filtered)}")

# Оставляем компании с funding_rounds > 0 для анализа раундов финансирования
df_funded = df_filtered[df_filtered['funding_rounds'] > 0].copy()

# Описательная статистика funding_rounds
print("Статистика по funding_rounds:")
print(df_funded['funding_rounds'].describe())

# Распределение по статусам
print("\nРаспределение компаний по статусам:")
print(df_funded['status'].value_counts())

# Среднее и медиана по статусу
rounds_mean = df_funded.groupby('status')['funding_rounds'].mean().reset_index()
rounds_median = df_funded.groupby('status')['funding_rounds'].median().reset_index()

rounds_mean = rounds_mean.sort_values(by='funding_rounds', ascending=False)

# Визуализация среднего количества раундов по статусу
plt.figure(figsize=(10, 6))
sns.barplot(data=rounds_mean, x='status', y='funding_rounds', palette='viridis')
plt.title('Среднее количество раундов финансирования по статусу')
plt.xlabel('Статус стартапа')
plt.ylabel('Среднее количество раундов')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Вывод сравнения среднего и медианы
print("\nСравнение среднего и медианы количества раундов по статусам:")
for _, row in rounds_mean.iterrows():
    median_val = rounds_median[rounds_median['status'] == row['status']]['funding_rounds'].values[0]
    print(f"Статус: {row['status']}, Среднее: {row['funding_rounds']:.2f}, Медиана: {median_val}")

print("\nПримечание: Медиана лучше отражает типичное значение при наличии выбросов.")

<span style="color:#2e8b57">Предварительный вывод</span>

Анализ показал, что стартапы со статусом IPO проходят в среднем почти 4 раунда финансирования, при этом медиана равна 3, что говорит о стабильном числе раундов перед выходом на биржу. Стартапы в статусе operating имеют среднее около 2,4 раундов, а медиану — 2, что отражает типичное количество привлечённых инвестиций.

Компании со статусом closed обычно проходят около 2 раундов (медиана — 1), а стартапы, которые были acquired — значительно меньше раундов, в среднем чуть больше 1, при медиане 0, что указывает на частые покупки на ранних этапах.

Важно отметить, что среднее значение чувствительно к выбросам, поэтому медиана даёт более надёжное представление о типичном числе раундов в каждой группе.


## Шаг 4. Итоговый вывод и рекомендации

Опишите, что было сделано в проекте, какие были сделаны выводы, подкрепляют ли они друг друга или заставляют сомневаться в полученных результатах.

<span style="color:#2e8b57">Пояснения к проекту</span>

# Итоговый вывод и рекомендации

В ходе проекта была проведена комплексная работа по изучению и предобработке нескольких датасетов, связанных со стартапами, их финансированием, приобретениями и сотрудниками.

<span style="color:#2e8b57">Что было сделано</span>

- **Знакомство с данными и предобработка**  
  Были загружены и проанализированы основные датасеты: `company_rounds`, `Acquisition`, `People`, `Education` и `Degrees`. В частности, для временных столбцов успешно применён тип `datetime`, что позволило эффективно работать с временными интервалами.  
  Проанализированы пропуски в данных — выявлены критические отсутствия в некоторых ключевых столбцах (например, даты закрытия компаний и детали раундов финансирования).

- **Исследование финансирования стартапов**  
  Рассмотрены раунды финансирования по годам, выявлены тенденции по объёму и количеству раундов. Проведён анализ средних и медианных значений количества раундов по статусам компаний.  
  Построены графики, которые визуализируют распределение раундов финансирования в зависимости от статуса (`IPO`, `operating`, `acquired`, `closed`).

- **Анализ категорий стартапов по цене и рискам**  
  Составлены топы категорий с высокими медианными ценами покупки и с большим разбросом цен, что помогает понять, какие направления привлекательны для инвесторов с разным уровнем риска.

- **Изучение данных о сотрудниках и образовании**  
  Проанализирована полнота данных о людях и их образовании. Выявлены существенные пропуски, что ограничивает возможности для полного анализа, но позволяет делать качественные выводы по имеющейся информации.

<span style="color:#2e8b57">Основные выводы</span>

- Данные о финансировании и закрытии компаний содержат значительные пропуски, что требует осторожности в их интерпретации. Медианные значения по раундам финансирования дают более стабильную оценку, чем средние, из-за влияния выбросов.

- Стартапы, которые выходят на IPO, проходят в среднем больше раундов финансирования, чем компании, которые были приобретены или закрыты. Это соответствует логике развития компаний.

- Категории с высокими ценами и большим разбросом свидетельствуют о высокой волатильности рынка стартапов, что подтверждает необходимость учитывать как потенциал доходности, так и риски инвестиций.

- Пропуски в данных о сотрудниках и образовании ограничивают возможность глубокого анализа влияния персонала на успех компаний, но выделяют направления для дальнейшего улучшения качества данных.

<span style="color:#2e8b57">Подтверждение и сомнения в результатах</span>

- Выводы, основанные на медианных значениях и распределениях по статусам, логично согласуются и подтверждают общие ожидания от поведения стартапов на рынке.

- Высокий уровень пропусков в ряде ключевых столбцов ставит под вопрос полноту и однозначность некоторых аналитических заключений, особенно связанных с временными аспектами и деталями раундов финансирования.

- Рекомендуется при дальнейшем анализе использовать методы работы с пропущенными данными, а также, по возможности, дополнить данные из внешних источников.

<span style="color:#2e8b57">Рекомендации</span>

- Провести дополнительную работу по очистке и заполнению пропусков в данных о финансировании и датах закрытия компаний.

- Использовать медианные показатели и устойчивые статистические методы при анализе финансовых данных.

- При анализе сотрудников и образования фокусироваться на подвыборках с наиболее полными данными.

- Для инвесторов и аналитиков учитывать как медианные значения финансирования, так и разбросы, чтобы балансировать риски и возможности.

- В дальнейшем рассмотреть расширение датасетов и интеграцию новых источников данных для повышения качества и полноты анализа.